# A4: cluster-theoretic Newton–Okounkov polytopes
An executable verification at **weight $2\rho$**, in four stages:
1. Generate all seeds and polytopes.
2. Compare combinatorial equivalence.
3. Compare affine unimodular equivalence and verify explicit affine maps.
4. Determine colored exchange-graph automorphisms.

Only the initial matrix and inequalities are input. The graph, polytopes,
classifications and permutations are computed here, without reading a stored
classification. Expected counts are assertions after calculation.

**Start here:** keep finite_type_verification.py beside this notebook and select
the **SageMath** kernel in Jupyter (tested with SageMath 10.8).
The code follows the seed enumeration and tropical-mutation workflow of
[0_functions.ipynb and 1_examples.ipynb](https://github.com/eunjeonglee-math/flag_NOpolytopes).

**Conventions.** Type $A_4$ means $\mathrm{SL}_5$; the mutable cluster type is
$D_6$. The initial word is $(1,2,1,3,2,1,4,3,2,1)$, with six mutable and four
frozen coordinates. The extended matrix is $10\times6$.
The initial anticanonical polytope is translated so each primitive facet has
constant one. H rows mean $c+a\cdot x\geq0$.

Seeds are identified by exact rational cluster functions modulo mutable relabeling;
frozen labels are fixed, and each identification checks the full extended matrix.
Matrix equality alone does not identify a seed.
Notebook seed IDs are BFS IDs, not manuscript Class numbers or BLMM labels.
The chronological sequence $[i_1,\ldots,i_k]$ means
$\mu_{i_k}\circ\cdots\circ\mu_{i_1}(\Delta_{t_0})$.

In [1]:
from pathlib import Path
import sys, time
from sage.all import QQ, ZZ, Graph, matrix, vector, factorial
from sage.env import SAGE_VERSION
HERE = Path.cwd().resolve()
HELPER = next((p for p in [HERE, HERE / "notebooks", *HERE.parents]
               if (p / "finite_type_verification.py").is_file()), None)
if HELPER is None:
    raise FileNotFoundError("Keep finite_type_verification.py beside this notebook.")
sys.path.insert(0, str(HELPER))
import finite_type_verification as verify
KIND = "A4"
RUN_LABEL = "reader-run"  # use a NEW name for a completely fresh computation
OUTPUT = HELPER / "verification-output" / (KIND + "-" + RUN_LABEL)
print("SageMath", SAGE_VERSION)
print("Output:", OUTPUT)

SageMath 10.8
Output: /mnt/c/Users/lej14/Dropbox/My/Sage/2024_Jeongsoo/A4_B2_reproducible_verification/verification-output/A4-reader-run


## 1. Generate all cluster-theoretic polytopes

Starting from the initial seed, we explore every possible mutation direction using breadth-first search (BFS). Newly discovered seeds are added to a queue for further exploration. The process stops when every discovered seed has been examined and no new seeds remain to be explored.

For each tropical mutation, split at $x_k=0$, map both pieces over $\mathbb Q$,
and check their convex hull on both halfspaces so no gap is silently filled.

In [3]:
initial = verify.initial_data(KIND)
print("Initial word:", initial["word"])
print("B ="); print(matrix(ZZ, initial["B"]))
print("H rows:")
for row in initial["H"]: print(row)
D = matrix.diagonal(ZZ, initial["symmetrizer"])
Bprincipal = matrix(ZZ, initial["B"])[:initial["rank"], :]
assert D * Bprincipal == -(D * Bprincipal).transpose()

Initial word: [1, 2, 1, 3, 2, 1, 4, 3, 2, 1]
B =
[ 0 -1  1  0  0  0]
[ 1  0 -1 -1  1  0]
[-1  1  0  0 -1  1]
[ 0  1  0  0 -1  0]
[ 0 -1  1  1  0 -1]
[ 0  0 -1  0  1  0]
[ 0  0  0  1  0  0]
[ 0  0  0 -1  1  0]
[ 0  0  0  0 -1  1]
[ 0  0  0  0  0 -1]
H rows:
[1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0]
[1, -1, 0, -1, 0, 0, -1, 0, 0, 0, -1]
[1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0]
[1, 0, -1, 0, 0, -1, 0, 0, 0, -1, 0]
[1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0]
[1, 0, 0, -1, 0, 0, -1, 0, 0, 0, -1]
[1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
[1, 0, 0, 0, -1, 0, 0, 0, -1, 0, 0]
[1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]
[1, 0, 0, 0, 0, -1, 0, 0, 0, -1, 0]
[1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0]
[1, 0, 0, 0, 0, 0, -1, 0, 0, 0, -1]
[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1]


In [4]:
start = time.monotonic()
catalogue, polytopes = verify.generate(KIND, OUTPUT)
N = len(polytopes)
assert catalogue["complete"] and N == 672
assert len(catalogue["directed_edges"]) == N * initial["rank"]
G = Graph([list(range(N)), catalogue["edges"]], format="vertices_and_edges")
print("Seeds:", N, "| undirected edges:", G.size(), "| dimension:", initial["dimension"])
print("Generation/load seconds:", round(time.monotonic()-start, 2))

Seeds processed: 50; discovered: 136 (0.2s)
Seeds processed: 100; discovered: 226 (0.4s)
Seeds processed: 150; discovered: 301 (0.6s)
Seeds processed: 200; discovered: 369 (0.8s)
Seeds processed: 250; discovered: 425 (1.1s)
Seeds processed: 300; discovered: 470 (1.3s)
Seeds processed: 350; discovered: 512 (1.6s)
Seeds processed: 400; discovered: 556 (1.9s)
Seeds processed: 450; discovered: 585 (2.1s)
Seeds processed: 500; discovered: 610 (2.3s)
Seeds processed: 550; discovered: 637 (2.5s)
Seeds processed: 600; discovered: 656 (2.7s)
Seeds processed: 650; discovered: 670 (2.9s)
Complete: 672 seeds; 2016 undirected edges.
Polytopes: 25/672 (11.9s)
Polytopes: 50/672 (21.2s)
Polytopes: 75/672 (34.4s)
Polytopes: 100/672 (46.5s)
Polytopes: 125/672 (62.0s)
Polytopes: 150/672 (76.8s)
Polytopes: 175/672 (95.6s)
Polytopes: 200/672 (110.7s)
Polytopes: 225/672 (130.4s)
Polytopes: 250/672 (144.9s)
Polytopes: 275/672 (160.3s)
Polytopes: 300/672 (174.6s)
Polytopes: 325/672 (194.5s)
Polytopes: 350/672

Choose a seed below to inspect its path, cluster variables, matrix, inequalities
and f-vector. Full f-vectors are computed only for the selected seed.
The classification uses exact incidence isomorphism, not f-vector equality.

In [5]:
seed_id = 0
seed = catalogue["seeds"][seed_id]
print("Seed", seed_id, "| path:", seed["sequence"])
print("Mutable cluster:", seed["cluster"][:initial["rank"]])
print("B ="); print(matrix(ZZ, seed["B"]))
print("f-vector:", polytopes[seed_id].f_vector())
print("H rows:")
for row in verify.h_rows(polytopes[seed_id]): print(row)

Seed 0 | path: []
Mutable cluster: ['x1', 'x2', 'x3', 'x4', 'x5', 'x6']
B =
[ 0 -1  1  0  0  0]
[ 1  0 -1 -1  1  0]
[-1  1  0  0 -1  1]
[ 0  1  0  0 -1  0]
[ 0 -1  1  1  0 -1]
[ 0  0 -1  0  1  0]
[ 0  0  0  1  0  0]
[ 0  0  0 -1  1  0]
[ 0  0  0  0 -1  1]
[ 0  0  0  0  0 -1]
f-vector: (1, 358, 2069, 5453, 8516, 8653, 5941, 2778, 870, 174, 20, 1)
H rows:
[1, -1, 0, -1, 0, 0, -1, 0, 0, 0, -1]
[1, 0, -1, 0, 0, -1, 0, 0, 0, -1, 0]
[1, 0, 0, -1, 0, 0, -1, 0, 0, 0, -1]
[1, 0, 0, 0, -1, 0, 0, 0, -1, 0, 0]
[1, 0, 0, 0, 0, -1, 0, 0, 0, -1, 0]
[1, 0, 0, 0, 0, 0, -1, 0, 0, 0, -1]
[1, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0]
[1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0]
[1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0]
[1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0]
[1, 0, 1, 0,

The BFS already checks every seed/matrix edge. The additional check below
recomputes the polytope on every directed edge, including coordinate relabeling.
It is enabled for B2; for A4 it is optional because it adds 4,032 polytope mutations.

In [6]:
CHECK_EVERY_POLYTOPE_EDGE = KIND == "B2"
geometric_edges_checked = 0
if CHECK_EVERY_POLYTOPE_EDGE:
    geometric_edges_checked = verify.verify_all_edges(catalogue, polytopes)
    print("Directed polytope edges verified:", geometric_edges_checked)
else:
    print("Seed/matrix edges checked. Optional all-edge polytope check not run.")

Seed/matrix edges checked. Optional all-edge polytope check not run.


## 2. Compare combinatorial equivalence
Different numbers of vertices or facets imply inequivalence.
Otherwise we use SageMath's is_combinatorially_isomorphic method with the
bipartite_graph algorithm, comparing vertex--facet incidence graphs.

In [7]:
combinatorial = verify.combinatorial_classes(polytopes, OUTPUT)
comb_groups = combinatorial["groups"]
assert len(comb_groups) == 162
print("Combinatorial classes:", len(comb_groups))
for label, group in enumerate(comb_groups, 1): print(label, group)
representatives = {group[0] for group in comb_groups}
cross_tests = [t for t in combinatorial["comparisons"]
               if t["source"] in representatives and t["target"] in representatives]
assert all(not t["equivalent"] for t in cross_tests)
print("Representative pairs requiring incidence tests:", len(cross_tests))

Combinatorial classification: 50/672; 22 classes
Combinatorial classification: 100/672; 47 classes
Combinatorial classification: 150/672; 78 classes
Combinatorial classification: 200/672; 101 classes
Combinatorial classification: 250/672; 118 classes
Combinatorial classification: 300/672; 145 classes
Combinatorial classification: 350/672; 158 classes
Combinatorial classification: 400/672; 162 classes
Combinatorial classification: 450/672; 162 classes
Combinatorial classification: 500/672; 162 classes
Combinatorial classification: 550/672; 162 classes
Combinatorial classification: 600/672; 162 classes
Combinatorial classification: 650/672; 162 classes
Combinatorial classes: 162
1 [0, 1, 31, 95, 388, 542, 662, 671]
2 [2, 3, 47, 60, 545, 546, 663, 664]
3 [4, 6, 9, 11, 645, 652, 668, 670]
4 [5, 10, 667, 669]
5 [7, 8, 629, 630]
6 [12, 17, 631, 633]
7 [13, 18, 632, 634]
8 [14, 21, 641, 644]
9 [15, 20, 637, 638]
10 [16, 19, 665, 666]
11 [22, 29, 654, 659]
12 [23, 30, 39, 46, 583, 604, 650, 65

In [8]:
## Check whether the polytopes are lattice polytopes or not. 
for seed_id, P in enumerate(polytopes):
    assert P.is_lattice_polytope(), f"Seed {seed_id}: not a lattice polytope"

## 3. Compare affine unimodular equivalence
Check that each polytope is integral with a lattice point at primitive facet distance
one. Translate this unique interior lattice point to zero and compute the complete
polar lattice normal form using palp_native. The complete forms, not hashes, are compared.

This is SageMath's built-in LatticePolytope.normal_form(), not a custom
equivalence criterion. It classifies linear lattice equivalence; centering first
reduces our affine question to that setting. We use the dual convention
$P^\vee=\{u:\langle u,x\rangle\geq-1\ \text{for all }x\in P\}$,
so the centered dual is the convex hull of the primitive inward facet normals.
Duality preserves equivalence (with inverse-transpose matrices), and working with
these much smaller vertex sets makes the A4 calculation practical.

If $U_P,U_Q$ send the centered polars to the same normal form, recover
$$A=(U_Q^{-1}U_P)^{-T},\qquad b=p_Q-Ap_P.$$
Check integrality, $\det A=\pm1$, the complete polar vertex correspondence, and
$$\{Av+b:v\in\operatorname{Vert}(P)\}=\operatorname{Vert}(Q).$$
Distinct normal forms certify inequivalence. This test is applied here at $2\rho$.

In [9]:
# A direct example of the built-in SageMath call used for every polytope.
from sage.all import LatticePolytope
initial_dual = LatticePolytope([h[1:] for h in verify.h_rows(polytopes[0])])
initial_normal_form = initial_dual.normal_form(algorithm="palp_native")
print("Sage normal form of the initial centered dual:")
print(matrix(ZZ, list(initial_normal_form)))

Sage normal form of the initial centered dual:
[ 1  0  0  0  0  0  0  0  0  0]
[ 0  1  0  0  0  0  0  0  0  0]
[ 0  0  1  0  0  0  0  0  0  0]
[ 0  0  0  1  0  0  0  0  0  0]
[ 0  0  0 -1  0  0  0  0  0  0]
[ 0  0 -1  0  0  0  0  0  0  0]
[ 0 -1  0  0  0  0  0  0  0  0]
[-1  0  0  0  0  0  0  0  0  0]
[ 0  0  0  0  1  0  0  0  0  0]
[ 0  0  0  0  0  1  0  0  0  0]
[ 0  0  0  0  0  0  1  0  0  0]
[ 0  0 -1 -1  0  0 -1  0  0  0]
[-1  0  0  1 -1  0  0  0  0  0]
[ 0 -1  1  0  0 -1  0  0  0  0]
[ 0  0  0  0  0  0  0  1  0  0]
[ 0  0  0  0  0  0  0  0  1  0]
[ 0 -1  1  0  0 -1  1  0 -1  0]
[-1  0 -1  0 -1  0 -1 -1  0  0]
[ 0  0  0  0  0  0  0  0  0  1]
[-1 -1  0  0 -1 -1  0 -1 -1 -1]


In [10]:
unimodular = verify.unimodular_classes(polytopes, OUTPUT)
uni_groups = unimodular["groups"]
assert len(uni_groups) == 162
assert all(c["all_vertices_verified"] and c["all_polar_vertices_verified"]
           for c in unimodular["certificates"])
print("Affine unimodular classes:", len(uni_groups))
print("Partitions coincide:", comb_groups == uni_groups)
assert (comb_groups == uni_groups) == True
for label, group in enumerate(uni_groups, 1): print(label, group)

Unimodular classification: 25/672; 13 classes
Unimodular classification: 50/672; 22 classes
Unimodular classification: 75/672; 34 classes
Unimodular classification: 100/672; 47 classes
Unimodular classification: 125/672; 59 classes
Unimodular classification: 150/672; 78 classes
Unimodular classification: 175/672; 84 classes
Unimodular classification: 200/672; 101 classes
Unimodular classification: 225/672; 106 classes
Unimodular classification: 250/672; 118 classes
Unimodular classification: 275/672; 129 classes
Unimodular classification: 300/672; 145 classes
Unimodular classification: 325/672; 153 classes
Unimodular classification: 350/672; 158 classes
Unimodular classification: 375/672; 162 classes
Unimodular classification: 400/672; 162 classes
Unimodular classification: 425/672; 162 classes
Unimodular classification: 450/672; 162 classes
Unimodular classification: 475/672; 162 classes
Unimodular classification: 500/672; 162 classes
Unimodular classification: 525/672; 162 classes
Un

Select any two seeds. If equivalent, display and reverify the exact affine map.
The default is a nontrivial pair in the first unimodular class.

In [11]:
source_seed, target_seed = uni_groups[0][:2]  # edit these IDs
certificate = verify.affine_map(polytopes[source_seed], polytopes[target_seed],
    unimodular["analyses"][source_seed], unimodular["analyses"][target_seed])
print("Source path:", catalogue["seeds"][source_seed]["sequence"])
print("Target path:", catalogue["seeds"][target_seed]["sequence"])
if certificate is None:
    print("Inequivalent: distinct complete polar normal forms.")
else:
    print("target_column = A * source_column + b")
    print("A ="); print(matrix(ZZ, certificate["A"]))
    print("b =", vector(ZZ, certificate["b"]))
    print("det(A) =", certificate["determinant"])
    print("All vertices verified:", certificate["all_vertices_verified"])

Source path: []
Target path: [1]
target_column = A * source_column + b
A =
[ 1  1  1  1  0  1  1  0  0  1]
[ 0 -1  0 -1 -1  0 -1  0 -1  0]
[ 0  0 -1  0 -1 -1  0 -1  0 -1]
[ 0  0  0  1  0  0  0  0  0  0]
[ 0  0  0  0  1  0  0  0  0  0]
[ 0  0  0  0  0  1  0  0  0  0]
[ 0  0  0  0  0  0  1  0  0  0]
[ 0  0  0  0  0  0  0  1  0  0]
[ 0  0  0  0  0  0  0  0  1  0]
[ 0  0  0  0  0  0  0  0  0  1]
b = (0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
det(A) = 1
All vertices verified: True


## 4. Determine colored exchange-graph automorphisms
Graph edges are unlabelled mutation edges. A color-preserving automorphism must
preserve **each class setwise**, rather than permuting the classes.
Compute G.automorphism_group(partition=groups), then independently filter all
uncolored graph automorphisms by their action on the computed classes.
Check every returned permutation on all graph vertices and edges.
Group orbits and polytope equivalence classes are different notions.

In [12]:
comb_symmetry = verify.colored_automorphisms(catalogue, comb_groups)
uni_symmetry = verify.colored_automorphisms(catalogue, uni_groups)
for label, result in [("combinatorial", comb_symmetry), ("affine unimodular", uni_symmetry)]:
    print(label)
    print("Full graph:", result["ambient_structure"], "order", result["ambient_order"])
    print("Colored:", result["colored_structure"], "order", result["colored_order"])
    print("Number of orbits:", len(result["orbits"]))
    for j,p in enumerate(result["generators"],1):
        print("Generator", j, "images of first seeds:", p[:min(N,12)])
        if KIND == "B2": print(verify.cycle_notation(p))
assert comb_symmetry["ambient_order"] == 24
assert comb_symmetry["colored_order"] == 4
assert uni_symmetry["colored_order"] == 4

combinatorial
Full graph: C2 x C2 x S3 order 24
Colored: C2 x C2 order 4
Number of orbits: 176
Generator 1 images of first seeds: [0, 1, 3, 2, 6, 5, 4, 8, 7, 11, 10, 9]
Generator 2 images of first seeds: [671, 662, 663, 664, 668, 667, 670, 629, 630, 645, 669, 652]
affine unimodular
Full graph: C2 x C2 x S3 order 24
Colored: C2 x C2 order 4
Number of orbits: 176
Generator 1 images of first seeds: [0, 1, 3, 2, 6, 5, 4, 8, 7, 11, 10, 9]
Generator 2 images of first seeds: [671, 662, 663, 664, 668, 667, 670, 629, 630, 645, 669, 652]


The polytope-class-preserving subgroup is $C_2\times C_2$ inside the
full group of order $24$. Its $176$ orbits refine the $162$ equivalence classes.
Some classes contain eight seeds, so not every equivalence is generated by this group.

In [13]:
assert len(cross_tests) == 47
assert comb_symmetry["colored_structure"] == "C2 x C2"
assert set(map(tuple,comb_symmetry["permutations"])) == set(map(tuple,uni_symmetry["permutations"]))
assert len(uni_symmetry["orbits"]) == 176
assert max(map(len,uni_groups)) == 8
print("162 polytope classes; 176 colored-group orbits; largest class size 8.")

162 polytope classes; 176 colored-group orbits; largest class size 8.


## Export the verification
Save all generated inequalities, paths, matrices, partitions, affine maps, normal
forms and colored-group permutations. The export states whether the optional
all-edge geometric check ran. Full f-vectors are not claimed for every seed.

In [14]:
report = dict(status="complete",kind=KIND,weight="2rho",sage_version=SAGE_VERSION,
    seed_count=N,edge_count=int(G.size()),coordinate_convention="column vectors",
    all_seed_matrix_edges_verified=True,geometric_directed_edges_checked=int(geometric_edges_checked),
    seeds=[dict(seed,H=verify.h_rows(P),vertex_count=int(P.n_vertices()),
                facet_count=int(P.n_facets())) for seed,P in zip(catalogue["seeds"],polytopes)],
    directed_edges=catalogue["directed_edges"],edges=catalogue["edges"],
    combinatorial=combinatorial,affine_unimodular=unimodular,
    combinatorial_symmetry=comb_symmetry,affine_unimodular_symmetry=uni_symmetry)
verify.save_json(OUTPUT/"verification.json",report)
print("Saved:", OUTPUT/"verification.json")

Saved: /mnt/c/Users/lej14/Dropbox/My/Sage/2024_Jeongsoo/A4_B2_reproducible_verification/verification-output/A4-reader-run/verification.json


## References and provenance
- [Original Hong–Kim–Lee notebooks](https://github.com/eunjeonglee-math/flag_NOpolytopes).
- [Fujita–Oya] Naoki Fujita and Hironori Oya, Newton–Okounkov polytopes of Schubert varieties arising from cluster structures, Transactions of the American Mathematical Society, Series B 12 (2025), 910–973. (https://doi.org/10.1090/btran/231) 
- [Sage lattice normal forms](https://doc.sagemath.org/html/en/reference/discrete_geometry/sage/geometry/lattice_polytope.html).
- [Sage combinatorial isomorphism](https://doc.sagemath.org/html/en/reference/discrete_geometry/sage/geometry/polyhedron/base4.html).

Copyright © Shin Hong, Jeongsoo Kim, Eunjeong Lee.
Prepared with assistance from ChatGPT.